<a href="https://colab.research.google.com/github/Aman6665/NLP-Program-using-Python-NLTK-and-spaCy/blob/main/Sequence_Classification_using_RNN_LSTM_(Keras_TensorFlow).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sequence Classification using LSTM (Keras/TensorFlow)
This notebook demonstrates how to build, train, and evaluate an LSTM-based recurrent neural network (RNN) for sequence classification using Keras and TensorFlow. We will generate synthetic sequence data for a binary classification task.

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Embedding
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

# Set random seed for reproducibility
np.random.seed(42)
tf.random.set_seed(42)
print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.21.0


### 1. Generate Synthetic Sequence Data
We will generate synthetic integer sequences. Let's assume we have a vocabulary of 1000 words (or tokens), and each sequence has a variable length. The task is to classify whether a sequence contains a specific pattern (e.g., if the sum of its tokens is even or odd, or if it contains certain high-value tokens).

In [2]:
# Parameters
vocab_size = 1000
num_samples = 2000
max_length = 50

# Generate random sequences of varying lengths
X_raw = [np.random.randint(1, vocab_size, size=np.random.randint(10, max_length)) for _ in range(num_samples)]

# Binary classification: 1 if sequence contains more than 3 values > 800, else 0
y = np.array([1 if np.sum(np.array(seq) > 800) > 3 else 0 for seq in X_raw])

# Pad sequences to ensure uniform length for training
X_padded = pad_sequences(X_raw, maxlen=max_length, padding='post', value=0)

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X_padded, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"Class distribution: {np.bincount(y)}")

X_train shape: (1600, 50)
X_test shape: (400, 50)
Class distribution: [ 467 1533]


### 2. Build the LSTM Model
We will construct a simple sequential model consisting of:
1. An `Embedding` layer to learn dense vector representations for our tokens.
2. An `LSTM` layer to capture the sequential dependencies.
3. A `Dense` output layer with a sigmoid activation function for binary classification.

In [3]:
embedding_dim = 64
lstm_units = 64

model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_length, mask_zero=True),
    LSTM(units=lstm_units, dropout=0.2, recurrent_dropout=0.2),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

model.summary()

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

### 3. Train the Model
We will fit the model using the training data and validate it on the test set.

In [4]:
history = model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=32,
    validation_data=(X_test, y_test)
)

Epoch 1/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 6s 64ms/step - accuracy: 0.7525 - loss: 0.5324 - val_accuracy: 0.7950 - val_loss: 0.3945
Epoch 2/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 3s 53ms/step - accuracy: 0.7825 - loss: 0.3695 - val_accuracy: 0.8075 - val_loss: 0.3910
Epoch 3/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 5s 96ms/step - accuracy: 0.8825 - loss: 0.2623 - val_accuracy: 0.8075 - val_loss: 0.4264
Epoch 4/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 3s 60ms/step - accuracy: 0.9256 - loss: 0.2001 - val_accuracy: 0.8250 - val_loss: 0.4477
Epoch 5/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 3s 52ms/step - accuracy: 0.9419 - loss: 0.1861 - val_accuracy: 0.8125 - val_loss: 0.4909
Epoch 6/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 5s 54ms/step - accuracy: 0.9631 - loss: 0.1305 - val_accuracy: 0.8125 - val_loss: 0.4973
Epoch 7/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 5s 53ms/step - accuracy: 0.9712 - loss: 0.1047 - val_accuracy: 0.8275 - val_loss: 0.5689
Epoch 8/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 3s 52ms/step - accuracy: 0.9781 - loss: 0.0839 - val_accuracy: 0.8400 - v

### 4. Evaluate the Model
Let's evaluate the performance of our model on the test dataset.

In [5]:
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

Test Loss: 0.6766
Test Accuracy: 0.8150
